In [1]:
from dataclasses import dataclass
import dask
import dask.dataframe as dd
from dask.distributed import Client, progress
from rich.console import Console
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
sys.path.append('./config')
from config import config

console = Console(highlight=False)
client = Client(n_workers=4, threads_per_worker=8)

In [2]:
def pshape(df):
    print(f'{df.shape[0]:,} rows')

In [3]:
data_path = config.data.staged / 'psychology'
if data_path.exists():
    console.print(f'[green] Found data path [/green]')
else:
    console.print(f'[red] Failed to find data path [/red]')

 Found data path 


In [4]:
df = dd.read_parquet(data_path, engine='fastparquet')
df = df.persist()
progress(df)
df = df.compute()
pshape(df)

[                                        ] | 0% Completed |  0.1s

[                                        ] | 0% Completed |  0.2s

[                                        ] | 0% Completed |  0.3s

[                                        ] | 0% Completed |  0.4s

[                                        ] | 0% Completed |  0.5s

...

[#####################                   ] | 54% Completed |  1.2s

[#######################                 ] | 57% Completed |  1.3s

[#############################           ] | 73% Completed |  1.4s

[########################################] | 100% Completed |  1.5s

2,064,088 rows


In [5]:
df = df[df['language'] == 'en']
df = df[df['type'] == 'article']
pshape(df)

1,440,797 rows


In [6]:
df['abstract_len'] = df['abstract'].apply(lambda x : len(x))
plt.figure()
plt.hist(df['abstract_len'])
plt.show()
stats = df['abstract_len'].describe()
print(stats)

count    1.440797e+06
mean     1.119304e+03
std      9.922367e+02
min      1.000000e+00
25%      6.460000e+02
50%      9.890000e+02
75%      1.385000e+03
max      4.301600e+04


Name: abstract_len, dtype: float64


In [7]:
# Filter out long/short abstracts (+- 3 std)
mean = stats['mean']
std = stats['std']
limit = std * 3
high = mean + limit
low = max(mean - limit, 300)
df = df[(df['abstract_len'] > low) & (df['abstract_len'] < high)]
pshape(df)

1,285,394 rows


In [8]:
# Filter abstracts contianing 'exclude' strings, remove elements of strings fitting 'remove' regex patterns 
@dataclass(frozen=True)
class Filter:
    exclude = ['keywords:', 'Keywords:' 'query=', 'http', 'Abstract', 'Abstract '
    'ADVERTISEMENT RETURN TO ISSUE', 'PAPER ACCEPTED FOR PUBLICATION'
    'Article Views', 'Altimetric-Citations', 
    'Copyright', 'copyright', '©'
    'reference to this paper', 'Google Scholar'
    # foreign connectives (remember to include space)
    'de ',
    
        # Vowels with accents
    'à', 'á', 'â', 'ã', 'ä', 'å', 'æ',
    'è', 'é', 'ê', 'ë',
    'ì', 'í', 'î', 'ï',
    'ò', 'ó', 'ô', 'õ', 'ö', 'ø', 'œ',
    'ù', 'ú', 'û', 'ü',
    'ý', 'ÿ',
    
    # Uppercase versions
    'À', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Æ',
    'È', 'É', 'Ê', 'Ë',
    'Ì', 'Í', 'Î', 'Ï',
    'Ò', 'Ó', 'Ô', 'Õ', 'Ö', 'Ø', 'Œ',
    'Ù', 'Ú', 'Û', 'Ü',
    'Ý',
    
    # Consonants with diacriticals
    'ç', 'Ç',
    'ñ', 'Ñ',
    'ð', 'Ð',
    'þ', 'Þ',
    'ß',

    ]
    tails = [
    'English', 'english',
    '<' , '>', ';', '@', '?', '[', ']', '{', '}'
    '#', '~', '/', '-', '_', '+', '=', '\\', '`', '¬', 
    '!', '£', '$', '%','^', '&', '*', '(', ')' 
    ]
    remove = []
@dataclass(frozen=True)
class Requirements:
    end_with = '.'

pshape(df)
filt = Filter()
df = df[~df['abstract'].str.contains('|'.join(filt.exclude))]
df = df[~df['abstract'].str.startswith('|'.join(filt.tails))]
df = df[~df['abstract'].str.endswith('|'.join(filt.tails))]

req = Requirements()
df = df[df['abstract'].str.endswith(req.end_with)]
pshape(df)

1,285,394 rows


1,032,765 rows


In [242]:
def sample(n, df, cols):
    mask = np.random.randint(0,df.shape[0]-1, (n,))
    samp = df.iloc[mask, :]
    return samp[cols].reset_index()
cols = ['abstract', 'abstract_len', 'cited_by_count', 'language']
samp = sample(10, df, cols)
for i in range(samp.shape[0]):
    print('\n======') 
    for c in cols:
        print(samp.loc[i, c])

This workshop will present an overview of the critical, practical and ethical issues that arise when psychotherapists work with divorcing clients and children. After presenting the big picture of the impact of divorce on society at large, the researched developmental needs of children in divorce will be contrasted with the many myths that have developed. This includes the extensive research on the best predictors of short-term and long-term outcomes of children of divorce and their implications for co-parenting and therapy following divorce. The dynamics of divorce, especially those with high conflict, will be graphically displayed. This will be followed by a presentation of twenty-five emerging roles for psychotherapists working with divorcing clients, along with the many ethical and clinical dilemmas with which a therapist must grapple. The importance of keeping clear boundaries between these roles will be strongly emphasized. Specific and practical guidelines (including protective l

1417


0


en


Frequently, students with emotional and behavior disorders (EBD) exhibit academic underachievement combined with high levels of externalizing behaviors and resistance to instructional efforts. Regardless of the present reading initiatives, research focusing on interventions for teaching reading to students with EBD continues to be limited. This article extends previous efforts to review literature concerning reading instruction interventions for students with EBD. Specifically, this review focuses on interventions employed in primary grades. Because of the paucity in research and documented issues related to late and misidentification of students with EBD, studies including students at risk of antisocial behaviors were included. Eleven studies were found and carefully reviewed. Results demonstrate the efficacy of several reading interventions, including Direct Instruction, peer tutoring, and behaviorally based procedures such as time delay prompting, trial and error, and differential r

...

en


SUMMARY Adenyl cyclase was measured in vitro in renal medullary homogenates from male mice of the Peru and CBA/FaCam strains. The basal activity of adenyl cyclase was increased on incubation with 30 m m -NaF and with varying concentrations of [8-arginine]-vasopressin (AVP) and [8-lysine]-vasopressin (LVP) up to 100 mu./ml. In both strains of mice, the maximal hormone activation was the same whichever vasopressin was used, and the same degree of stimulation was observed on incubation of homogenates with both hormones together. It is concluded that both hormones have the same intrinsic activity in this system, and are acting on the same population of receptors within each strain of mouse. Half-maximal adenyl cyclase activation was achieved with 240 ± 50 ( s.e.m. ) μu. AVP/ml and 920 ± 160 μu. LVP/ml in homogenates from CBA/FaCam mice; and 240 ± 40 μu. AVP/ml and 1900 ± 250 μu. LVP/ml in Peru mice. These results are compared with previously reported potencies in these mice of the two vaso

1387


1


en


In [9]:
df_out = dd.from_pandas(df, npartitions=64)
df_out = dd.to_parquet(
    df_out,
    str(config.data.staged / 'psychology_clean'),
    engine = 'pyarrow',
    compression = 'zstd',
    compression_level = 1,
    write_statistics = True,
    compute = False,
    overwrite = True,
)
progress(client.compute(df_out))
print('done')

[                                        ] | 0% Completed |  8.3s

[                                        ] | 0% Completed |  8.4s

[                                        ] | 0% Completed |  8.6s

[                                        ] | 0% Completed |  8.7s

[                                        ] | 0% Completed |  8.8s

...

[################################        ] | 80% Completed |  9.9s

[######################################  ] | 96% Completed | 10.0s

[####################################### ] | 97% Completed | 10.1s

[########################################] | 100% Completed | 10.2s

done
